In [1]:
"""
한국 매출 예측 결과 분석 스크립트

DB에 저장된 예측 결과를 분석하여:
1. Unique ticker 추출
2. Ticker별 매출 증가율 계산 및 순위 매기기
"""

import pandas as pd
import pymysql
import socket
from typing import Dict, Tuple


def get_db_host():
    """현재 머신에 따라 DB 호스트 자동 선택"""
    hostname = socket.gethostname()
    if 'desktop' in hostname.lower():
        return '192.168.0.8'
    else:
        return 'localhost'


def get_db_config():
    """DB 연결 정보 반환"""
    return {
        'user': 'stox7412',
        'password': 'Apt106503!~',
        'host': get_db_host(),
        'port': 3307,
        'database': 'investar'
    }


def get_unique_tickers(db_info: dict, table_name: str = "korea_revenue_forecast_result") -> pd.DataFrame:
    """
    DB에서 unique ticker 목록 조회

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    table_name : str
        테이블 이름

    Returns:
    --------
    pd.DataFrame
        ticker와 개수를 포함한 DataFrame
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT
            ticker,
            COUNT(*) as record_count,
            MIN(date) as first_date,
            MAX(date) as last_date
        FROM {table_name}
        GROUP BY ticker
        ORDER BY ticker
        """

        df = pd.read_sql(sql, conn)

        print("=" * 70)
        print("📊 Unique Ticker 조회 결과")
        print("=" * 70)
        print(f"✓ 총 ticker 개수: {len(df):,}개")
        print(f"✓ 총 레코드 수: {df['record_count'].sum():,}개")
        print("=" * 70)

        return df

    finally:
        conn.close()

def calculate_revenue_growth_rank(
    db_info: dict,
    table_name: str = "korea_revenue_forecast_result",
    indicator: str = "Ensemble",
    start_date: str = None,
    end_date: str = None,
    top_n: int = None
) -> pd.DataFrame:
    """
    Ticker별 매출 증가율을 계산하고 순위를 매김

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    table_name : str
        테이블 이름
    indicator : str
        사용할 지표 (SARIMA, ETS, Theta, Ensemble)
    start_date : str, optional
        시작 날짜 (YYYY-MM-DD 형식, None이면 가장 빠른 날짜)
    end_date : str, optional
        종료 날짜 (YYYY-MM-DD 형식, None이면 가장 늦은 날짜)
    top_n : int, optional
        상위 n개만 반환 (None이면 전체)

    Returns:
    --------
    pd.DataFrame
        ticker별 증가율과 순위 정보
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 날짜 필터링 조건 추가
        date_filter = ""
        params = [indicator]

        if start_date is not None:
            date_filter += " AND date >= %s"
            params.append(start_date)

        if end_date is not None:
            date_filter += " AND date <= %s"
            params.append(end_date)

        # 전체 데이터 조회
        sql = f"""
        SELECT
            date,
            ticker,
            indicator,
            value
        FROM {table_name}
        WHERE indicator = %s{date_filter}
        ORDER BY ticker, date
        """

        df = pd.read_sql(sql, conn, params=params)

        if df.empty:
            print(f"⚠️ {indicator} 데이터가 없습니다.")
            return pd.DataFrame()

        # 날짜 컬럼을 datetime으로 변환
        df['date'] = pd.to_datetime(df['date'])

        # Ticker별 증가율 계산
        result_list = []

        for ticker in df['ticker'].unique():
            ticker_df = df[df['ticker'] == ticker].sort_values('date')

            if len(ticker_df) < 2:
                continue

            # 첫 번째와 마지막 값
            first_value = ticker_df.iloc[0]['value']
            last_value = ticker_df.iloc[-1]['value']

            # 첫 번째와 마지막 날짜
            first_date = ticker_df.iloc[0]['date']
            last_date = ticker_df.iloc[-1]['date']

            # 증가율 계산 (%)
            if first_value > 0:
                growth_rate = ((last_value - first_value) / first_value) * 100
            else:
                growth_rate = None

            # 절대 증가액
            growth_amount = last_value - first_value

            result_list.append({
                'ticker': ticker,
                'first_date': first_date,
                'last_date': last_date,
                'first_value': first_value,
                'last_value': last_value,
                'growth_amount': growth_amount,
                'growth_rate_pct': growth_rate,
                'indicator': indicator
            })

        # DataFrame 생성
        result_df = pd.DataFrame(result_list)

        if result_df.empty:
            print("⚠️ 증가율을 계산할 수 있는 데이터가 없습니다.")
            return result_df

        # 증가율 기준 순위 매기기 (내림차순)
        result_df = result_df.sort_values('growth_rate_pct', ascending=False)
        result_df['rank'] = range(1, len(result_df) + 1)

        # 컬럼 순서 조정
        result_df = result_df[[
            'rank', 'ticker', 'growth_rate_pct', 'growth_amount',
            'first_value', 'last_value', 'first_date', 'last_date', 'indicator'
        ]]

        # 결과 요약 출력
        period_info = ""
        if start_date or end_date:
            period_info = f" (기간: {start_date or '처음'} ~ {end_date or '끝'})"

        print("\n" + "=" * 70)
        print(f"📈 매출 증가율 분석 결과 ({indicator}){period_info}")
        print("=" * 70)
        print(f"✓ 분석 대상 ticker: {len(result_df):,}개")
        print(f"✓ 실제 데이터 기간: {result_df['first_date'].min()} ~ {result_df['last_date'].max()}")
        print(f"✓ 평균 증가율: {result_df['growth_rate_pct'].mean():.2f}%")
        print(f"✓ 중간 증가율: {result_df['growth_rate_pct'].median():.2f}%")
        print("=" * 70)

        # Top N만 반환
        if top_n is not None:
            result_df = result_df.head(top_n)

        return result_df

    finally:
        conn.close()

def analyze_all_indicators(
    db_info: dict,
    table_name: str = "korea_revenue_forecast_result",
    top_n: int = 20
) -> Dict[str, pd.DataFrame]:
    """
    모든 지표(SARIMA, ETS, Theta, Ensemble)에 대해 증가율 분석

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    table_name : str
        테이블 이름
    top_n : int
        각 지표별 상위 n개

    Returns:
    --------
    dict
        지표별 분석 결과 딕셔너리
    """
    indicators = ['SARIMA', 'ETS', 'Theta', 'Ensemble']
    results = {}

    for indicator in indicators:
        print(f"\n{'='*70}")
        print(f"분석 중: {indicator}")
        print(f"{'='*70}")

        df = calculate_revenue_growth_rank(
            db_info=db_info,
            table_name=table_name,
            indicator=indicator,
            top_n=top_n
        )

        results[indicator] = df

    return results


def save_analysis_results(results: Dict[str, pd.DataFrame], output_prefix: str = "revenue_growth_analysis"):
    """
    분석 결과를 CSV 파일로 저장

    Parameters:
    -----------
    results : dict
        지표별 분석 결과
    output_prefix : str
        출력 파일명 접두사
    """
    for indicator, df in results.items():
        if not df.empty:
            filename = f"{output_prefix}_{indicator}.csv"
            df.to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"✅ 저장 완료: {filename} ({len(df)}행)")


def main():
    """메인 실행 함수"""

    print("\n" + "=" * 80)
    print("한국 매출 예측 결과 분석")
    print("=" * 80)

    # 1. DB 연결 정보
    db_info = get_db_config()
    print(f"\n✅ DB 연결 정보 설정 완료 (host: {db_info['host']})")

    # 2. Unique ticker 조회
    print("\n" + "=" * 80)
    print("1단계: Unique Ticker 조회")
    print("=" * 80)

    unique_tickers = get_unique_tickers(db_info)

    if not unique_tickers.empty:
        print("\n📊 Ticker 샘플 (처음 10개):")
        print(unique_tickers.head(10).to_string(index=False))

        # 전체 ticker 목록 저장
        unique_tickers.to_csv('unique_tickers_list.csv', index=False, encoding='utf-8-sig')
        print(f"\n✅ Ticker 목록 저장: unique_tickers_list.csv")

    # 3. 매출 증가율 순위 분석
    print("\n" + "=" * 80)
    print("2단계: 매출 증가율 순위 분석")
    print("=" * 80)

    print("\n분석 옵션:")
    print("1. Ensemble만 분석 (추천)")
    print("2. 모든 지표 분석 (SARIMA, ETS, Theta, Ensemble)")

    choice = input("\n선택 (1 또는 2, 기본값=1): ").strip() or "1"

    if choice == "2":
        # 모든 지표 분석
        results = analyze_all_indicators(db_info, top_n=100)

        # 결과 저장
        print("\n" + "=" * 80)
        print("결과 저장 중...")
        print("=" * 80)
        save_analysis_results(results)

        # Ensemble 결과 출력
        if 'Ensemble' in results and not results['Ensemble'].empty:
            print("\n" + "=" * 80)
            print("📊 Ensemble 기준 상위 20개 Ticker")
            print("=" * 80)
            print(results['Ensemble'].head(20).to_string(index=False))
    else:
        # Ensemble만 분석
        ensemble_result = calculate_revenue_growth_rank(
            db_info=db_info,
            indicator="Ensemble",
            top_n=100
        )

        if not ensemble_result.empty:
            # 결과 저장
            ensemble_result.to_csv(
                'revenue_growth_analysis_Ensemble.csv',
                index=False,
                encoding='utf-8-sig'
            )
            print(f"\n✅ 저장 완료: revenue_growth_analysis_Ensemble.csv ({len(ensemble_result)}행)")

            # 상위 20개 출력
            print("\n" + "=" * 80)
            print("📊 Ensemble 기준 상위 20개 Ticker")
            print("=" * 80)
            print(ensemble_result.head(20).to_string(index=False))

            # 하위 20개도 출력
            print("\n" + "=" * 80)
            print("📊 Ensemble 기준 하위 20개 Ticker")
            print("=" * 80)
            print(ensemble_result.tail(20).to_string(index=False))

    print("\n" + "=" * 80)
    print("✅ 분석 완료!")
    print("=" * 80)


# if __name__ == "__main__":
#     try:
#         main()
#     except KeyboardInterrupt:
#         print("\n\n⚠️ 사용자에 의해 중단되었습니다.")
#     except Exception as e:
#         print(f"\n\n❌ 오류 발생: {e}")
#         import traceback
#         traceback.print_exc()

In [3]:
from DATA.stock_invest_function import fetch_table_data, get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}


# unique_tickers = get_unique_tickers(db_info)

In [9]:
# 2024년 한 해 동안의 성장률
result = calculate_revenue_growth_rank(
    db_info=db_info,
    indicator="ETS",
    start_date="2026-01-01",
    end_date="2026-12-31",
    top_n=30
)



📈 매출 증가율 분석 결과 (ETS) (기간: 2026-01-01 ~ 2026-12-31)
✓ 분석 대상 ticker: 1,091개
✓ 실제 데이터 기간: 2026-01-01 00:00:00 ~ 2026-12-31 00:00:00
✓ 평균 증가율: 7.01%
✓ 중간 증가율: 4.32%


In [10]:
result

,rank,ticker,growth_rate_pct,growth_amount,first_value,last_value,first_date,last_date,indicator
66,1,001470,887.756399,1.039898e+11,1.171378e+10,1.157036e+11,2026-03-31,2026-12-31,ETS
882,2,056360,303.941952,2.519623e+10,8.289817e+09,3.348605e+10,2026-03-31,2026-12-31,ETS
807,3,049470,282.477906,7.808125e+09,2.764154e+09,1.057228e+10,2026-03-31,2026-12-31,ETS
880,4,056080,273.164647,1.122017e+09,4.107477e+08,1.532765e+09,2026-03-31,2026-12-31,ETS
500,5,021080,219.887873,2.957376e+10,1.344947e+10,4.302323e+10,2026-03-31,2026-12-31,ETS
1002,6,078000,213.196927,1.318685e+10,6.185290e+09,1.937214e+10,2026-03-31,2026-12-31,ETS
934,7,065420,196.123875,3.434009e+09,1.750939e+09,5.184948e+09,2026-03-31,2026-12-31,ETS
233,8,005870,193.855079,6.263595e+10,3.231071e+10,9.494666e+10,2026-03-31,2026-12-31,ETS
638,9,035000,176.741294,1.288921e+11,7.292701e+10,2.018192e+11,2026-03-31,2026-12-31,ETS
728,10,040610,168.920776,1.082449e+10,6.408029e+09,1.723252e+10,2026-03-31,2026-12-31,ETS
